# Aurora — calcul natif Colab

VM de calcul : matériel mesuré, lot natif empreinté, budget explicite et export vérifié. Un GPU G4 ou une capacité donnée ne sont jamais supposés.

Sources Python natives incluses et empreintées, sans poids ni identifiants. Exécuter les cellules dans l’ordre ; la cellule de dépendances installe le navigateur requis. Aucun entraînement ni promotion automatique. Pour entraîner un moteur non inclus dans ce banc, fournir son checkout vérifié complet et le lot natif correspondant.


In [ ]:
from pathlib import Path, PurePosixPath
import base64, hashlib, io, json, os, stat, sys, zipfile
def extract_bundle(raw, destination, expected_sha256, max_bytes=100 * 1024 * 1024):
    """Validate all paths and hashes before writing; preserve any different existing file."""
    if len(raw) > max_bytes or hashlib.sha256(raw).hexdigest() != expected_sha256:
        raise ValueError("Source bundle hash or size mismatch")
    destination = Path(destination).resolve()
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        names = archive.namelist()
        if len(set(names)) != len(names) or sum(i.file_size for i in archive.infolist()) > max_bytes:
            raise ValueError("Invalid or oversized source bundle")
        manifest = json.loads(archive.read("source_manifest.json"))
        if not isinstance(manifest, dict) or set(names) != set(manifest) | {"source_manifest.json"}:
            raise ValueError("Source manifest does not match archive")
        pending = []
        for info in archive.infolist():
            relative = PurePosixPath(info.filename)
            if (str(relative) != info.filename or relative.is_absolute() or ".." in relative.parts or "\\" in info.filename
                    or stat.S_ISLNK(info.external_attr >> 16)
                    or (info.filename != "source_manifest.json" and relative.suffix != ".py")):
                raise ValueError("Unsafe source bundle member")
            target = destination.joinpath(*relative.parts)
            if not target.resolve().is_relative_to(destination):
                raise ValueError("Source bundle escapes destination")
            content = archive.read(info)
            if info.filename in manifest and hashlib.sha256(content).hexdigest() != manifest[info.filename]:
                raise ValueError("Source file hash mismatch")
            if target.exists() and target.read_bytes() != content:
                raise ValueError("Existing different source preserved")
            pending.append((target, content))
        for target, content in pending:
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists():
                with target.open("xb") as stream:
                    stream.write(content)
    return destination

SOURCE_SHA256 = '375fe3a8d571b9ffe8a98f2d3d3011a9fa2b9aba4ad2563818cb454abdb5f17b'
source_bytes = base64.b64decode('...')
# A unique, hash-named directory keeps source revisions separate.
REPO = extract_bundle(source_bytes, Path.cwd() / "aurora_native_sources" / SOURCE_SHA256[:16], SOURCE_SHA256)
sys.path.insert(0, str(REPO))
print("Native source files verified:", 38)


In [ ]:
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError
# Reuse the VM's numerical stack; install only the missing browser dependencies.
requirements = {"playwright": "1.61.0", "psutil": "7.2.2"}
missing = []
for name, expected in requirements.items():
    try:
        installed = version(name)
    except PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(f"{name}=={expected}")
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True, timeout=180)
subprocess.run([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"], check=True, timeout=240)


In [ ]:
from pathlib import Path
import os, sys, json
from auto_rl.remote_lab import probe_resources, validate_native, run_payload
print(json.dumps(probe_resources(), indent=2))


In [ ]:
# Fill from a package produced by Aurora's native cloud pipeline.
# Keep False for preflight. Training is a separate, deliberate cell action.
RUN_TRAINING = False
JOB_ID = ""
PAYLOAD_SHA256 = ""
INPUTS = Path.cwd() / "aurora_inputs"
REQUIREMENTS_FILE = INPUTS / "measured_requirements.json"
SESSION_SECONDS = None  # Your explicit budget, including checkpoint/export margin.
if RUN_TRAINING:
    assert SESSION_SECONDS is not None and SESSION_SECONDS > 0
    requirements = json.loads(REQUIREMENTS_FILE.read_text())
    receipt = run_payload(job_id=JOB_ID, payload_hash=PAYLOAD_SHA256,
                          inputs=INPUTS, output=Path.cwd() / "aurora_experiments",
                          requirements=requirements, session_seconds=SESSION_SECONDS)
    print(json.dumps(receipt, indent=2))
else:
    print("Preflight only. No training started and no model promoted.")
